# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saadtalat111/flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The Queue: High Rank, Low CTR
This playbook provides a ranked, directional queue for content teams. It flags pages that have already earned top-tier Google visibility but are failing to capture expected clicks.
Primary Action: UPDATE_META (Rewrite the Title Tag and Meta Description).
Reason Code: HIGH_RANK_LOW_CTR.
Archetype-to-Action Mapping: Pages are ranked by a composite penalty score: (6 - avg_position) * (3.0 - ctr). The highest scores represent pages that rank highest (e.g., position 1-2) but have the lowest CTRs (e.g., < 1.0%), making them the most urgent candidates for optimization.

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# 1. Load the dataset (using the direct GitHub raw URL)
repo_url = 'https://raw.githubusercontent.com/saadtalat111/flyrank/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(repo_url)

# 2. Apply baseline logic to create the queue
df_clean = df[(df['avg_position'] > 0) & (df['avg_position'] <= 5)].copy()
df_clean['is_critical_ctr'] = (df_clean['ctr'] < 3.0).astype(int)

# 3. Calculate the baseline score for ranking
df_clean['action_score'] = (6 - df_clean['avg_position']) * (3.0 - df_clean['ctr'])
df_clean.loc[df_clean['ctr'] >= 3.0, 'action_score'] = 0

# 4. Filter for actionable items and rank
playbook_queue = df_clean[df_clean['action_score'] > 0].sort_values(by='action_score', ascending=False)
playbook_queue['reason_code'] = 'HIGH_RANK_LOW_CTR'
playbook_queue['action'] = 'UPDATE_META'

print("--- Playbook Queue Generated ---")
print(f"Total pages flagged for review: {len(playbook_queue)}")


--- Playbook Queue Generated ---
Total pages flagged for review: 3672


## 2. Intended use and limits

Intended Use:
This queue is a decision-support tool for SEO managers and editors. It is intended to guide sprint planning by identifying which pages to review first. It highlights where CTR optimization is most likely to yield traffic lift because the page visibility already exists.
Limits:
This analysis relies on cross-sectional observational data. We measured an association between position and expected CTR; we have not run a controlled experiment proving that rewriting a specific title tag causes a CTR increase. Furthermore, the math formula breaks down on tracking anomalies (e.g., average positions of 0.1 or 0.2), which represent knowledge panels or sitelinks rather than standard organic results.

In [2]:
# Demonstrating the limit: separating out the tracking anomalies
anomalies = playbook_queue[playbook_queue['avg_position'] < 1.0]
valid_targets = playbook_queue[playbook_queue['avg_position'] >= 1.0]

print("--- Limit Validation ---")
print(f"Valid standard search targets (Pos >= 1.0): {len(valid_targets)}")
print(f"Anomalies requiring exclusion/special review (Pos < 1.0): {len(anomalies)}")


--- Limit Validation ---
Valid standard search targets (Pos >= 1.0): 3596
Anomalies requiring exclusion/special review (Pos < 1.0): 76


## 3. Human review + the no-go list

Human Review Rules:
Before executing an UPDATE_META action, a human editor must search the target query manually to verify the SERP layout.
Is a competitor using a rich snippet (like stars or images) that steals clicks?
Is the user's intent purely definitional, meaning they get their answer directly from the search results page without needing to click?
The No-Go List (What NOT to automate):
Do not automate metadata deployment: AI can draft new title tags, but they must not be auto-published without a human confirming they accurately reflect the page content.
Do not optimize sub-1.0 positions: Pages with an average position less than 1.0 are excluded from automated workflows as they are tracking artifacts or zero-click widgets.

In [3]:
# Generating a clean view for the human reviewer
review_columns = ['content_id', 'avg_position', 'ctr', 'word_count', 'reason_code', 'action']
human_review_queue = valid_targets[review_columns].head(5)

print("--- Sample Human Review View ---")
print(human_review_queue)


--- Sample Human Review View ---
                 content_id  avg_position  ctr  word_count        reason_code  \
14661  content_2e6955a6d440           1.0  0.0      2582.0  HIGH_RANK_LOW_CTR   
22986  content_0b5377579ec0           1.0  0.0      4152.0  HIGH_RANK_LOW_CTR   
19146  content_d716815640bf           1.0  0.0      3742.0  HIGH_RANK_LOW_CTR   
25406  content_ba36299c8f27           1.0  0.0         NaN  HIGH_RANK_LOW_CTR   
27508  content_643711d5a1ef           1.0  0.0      2785.0  HIGH_RANK_LOW_CTR   

            action  
14661  UPDATE_META  
22986  UPDATE_META  
19146  UPDATE_META  
25406  UPDATE_META  
27508  UPDATE_META  


## 4. Monitoring / retrain triggers

Monitoring & Stale Triggers:
Because this playbook relies on a deterministic rule rather than a learned model, "retraining" is not required. However, the rule's thresholds (e.g., Top 5 position and <3.0% CTR) will go stale if the underlying environment changes.
Triggers to pause the playbook and re-evaluate:
A major Google SERP layout update: If Google introduces widespread zero-click widgets, the baseline expectation for page-one CTR will collapse, and the 3.0% threshold will need to be lowered.
Base Rate Shift: If the overall portfolio CTR drops by more than 10% month-over-month, the queue will flag too many false positives, signaling that the threshold must be adjusted.

In [4]:
# Documenting current base rates so we know if they shift in the future
current_median_ctr_top5 = valid_targets['ctr'].median()
current_critical_flag_rate = len(valid_targets) / len(df_clean)

print("--- Current Benchmarks (for future monitoring) ---")
print(f"Median CTR for flagged valid pages: {current_median_ctr_top5:.2f}%")
print(f"Percentage of Top-5 pages currently flagged: {current_critical_flag_rate:.1%}")


--- Current Benchmarks (for future monitoring) ---
Median CTR for flagged valid pages: 0.15%
Percentage of Top-5 pages currently flagged: 91.7%


## 5. Exports for the paper

Exports Generated:
The Ranked Queue (CSV): Exported to work/outputs/action_playbook_queue.csv. This contains only valid targets (position >= 1.0) ready for the content team.
Score Distribution (Figure): Exported to work/figures/queue_score_distribution.png to be used in the final research paper to visualize the optimization priority.

In [5]:
# 1. Setup Directories
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# 2. Export the final, valid queue to CSV
csv_path = 'work/outputs/action_playbook_queue.csv'
valid_targets.to_csv(csv_path, index=False)
print(f"Exported queue to: {csv_path}")

# 3. Generate and export a figure for the final paper
plt.figure(figsize=(8, 5))
plt.hist(valid_targets['action_score'], bins=20, color='rebeccapurple', edgecolor='black')
plt.title('Distribution of Optimization Priority Scores')
plt.xlabel('Action Score (Higher = More Urgent)')
plt.ylabel('Number of Pages')
plt.grid(axis='y', alpha=0.75)

fig_path = 'work/figures/queue_score_distribution.png'
plt.savefig(fig_path, bbox_inches='tight')
plt.close()
print(f"Exported figure to: {fig_path}")


Exported queue to: work/outputs/action_playbook_queue.csv
Exported figure to: work/figures/queue_score_distribution.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.